# Chapter 19 — When OLS Goes Wrong

Companion code for *Python for Indian Economics Students*. The explanations, exercises and discussion are in the book.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slxnjdhv/python-for-indian-economics/blob/main/labs/ch19_when_ols_goes_wrong.ipynb)

In [ ]:
# Run this cell first on Google Colab (not needed on your own computer
# if you installed requirements.txt).
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/slxnjdhv/python-for-indian-economics.git

## 19.1 The baseline model

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

from indiadata import read_plfs

HH_KEYS = ["qtr", "visit", "sec", "st", "mfsu", "seg", "sss", "ssu"]
p = read_plfs("perv1", columns=["seg", "sss", "ssu", "sex", "age", "marst",
                                "form_edu", "acws", "ern_reg"])
p["hh"] = p.groupby(HH_KEYS, observed=True).ngroup()       # household number
p["kids05"] = (p["age"] <= 5).groupby(p["hh"]).transform("sum")
print(f"{p['hh'].nunique():,} households, {len(p):,} persons")

In [ ]:
s = p[(p["acws"] == 31) & (p["ern_reg"] > 0)
      & p["age"].between(15, 59) & p["sex"].isin([1, 2])]
d = pd.DataFrame({
    "earn": s["ern_reg"].astype(float),
    "educ": s["form_edu"].astype(float),
    "age": s["age"].astype(float),
    "female": (s["sex"] == 2).astype(float),
    "urban": (s["sec"] == 2).astype(float),
    "st": s["st"].astype(int),
    "mfsu": s["mfsu"].astype(int),
})
d["lwage"] = np.log(d["earn"])
d["exper"] = (d["age"] - d["educ"] - 6).clip(lower=0)

MINCER = "lwage ~ educ + exper + I(exper**2) + female + urban"
m = smf.ols(MINCER, data=d).fit()
print(m.summary().tables[1])

## 19.2 Heteroskedasticity

In [ ]:
spread = (d.assign(resid=m.resid).groupby("educ")["resid"]
          .agg(["std", "size"]).query("size >= 50"))

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.bar(spread.index, spread["std"], color="#9db7d5", width=0.8)
ax.axhline(np.sqrt(m.scale), color="#c55a11", ls="--", lw=1)
ax.text(spread.index.max() + 0.6, np.sqrt(m.scale), "overall", color="#c55a11",
        fontsize=9, va="center")
ax.set_xlabel("Years of formal education")
ax.set_ylabel("SD of OLS residuals")
ax.set_title("The spread of log earnings is not constant: heteroskedasticity",
             loc="left", fontsize=13)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.06, "Residuals from the Mincer regression; years with fewer than "
         "50 workers omitted. Source: author's calculation from PLFS 2023–24 "
         "unit-level data (NSO, MoSPI).", fontsize=8, color="grey")
plt.show()
print("SD of residuals at selected years of schooling:")
print(spread["std"].loc[[0, 6, 10, 12, 15, 17, 21]].round(2).to_string())

In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan, het_white

lm, lm_p, _, _ = het_breuschpagan(m.resid, m.model.exog)
print(f"Breusch–Pagan: LM = {lm:,.0f}, p = {lm_p:.3g}")

with warnings.catch_warnings():          # female² = female: harmless duplicate
    warnings.simplefilter("ignore")
    lm_w, lm_w_p, _, _ = het_white(m.resid, m.model.exog)
print(f"White:         LM = {lm_w:,.0f}, p = {lm_w_p:.3g}")

In [ ]:
se = pd.DataFrame({c: smf.ols(MINCER, data=d).fit(cov_type=c).bse
                   for c in ["nonrobust", "HC0", "HC1", "HC2", "HC3"]})
se.round(5)

## 19.3 Clustered standard errors: neighbours are not independent

In [ ]:
fits = {
    "classical": smf.ols(MINCER, data=d).fit(),
    "robust (HC1)": smf.ols(MINCER, data=d).fit(cov_type="HC1"),
    "clustered by FSU": smf.ols(MINCER, data=d).fit(
        cov_type="cluster", cov_kwds={"groups": d["mfsu"]}),
    "clustered by state": smf.ols(MINCER, data=d).fit(
        cov_type="cluster", cov_kwds={"groups": d["st"]}),
}
se_cl = pd.DataFrame({k: f.bse for k, f in fits.items()})
ratio = (se_cl.div(se_cl["classical"], axis=0)).loc[["educ", "female", "urban"]]
print(f"Clusters: {d['mfsu'].nunique():,} FSUs, {d['st'].nunique()} states")
display(se_cl.round(5))
print("SE relative to classical:")
ratio.round(2)

## 19.4 Multicollinearity

In [ ]:
m_bad = smf.ols("lwage ~ educ + exper + age + female + urban", data=d).fit()
print(m_bad.params.round(4))
print(f"Condition number of X: {np.linalg.cond(m_bad.model.exog):,.0f}")
X_bad = m_bad.model.exog
r2_age = sm.OLS(X_bad[:, 3], np.delete(X_bad, 3, axis=1)).fit().rsquared
print(f"R² of age on the other regressors: {r2_age:.4f} "
      f"(so its VIF is {1 / (1 - r2_age):,.0f})")

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor


def vif_table(fit):
    X = fit.model.exog
    names = fit.model.exog_names
    return pd.Series([variance_inflation_factor(X, i) for i in range(1, X.shape[1])],
                     index=names[1:], name="VIF").round(2)


d["exper_c"] = d["exper"] - d["exper"].mean()          # centred experience
m_c = smf.ols("lwage ~ educ + exper_c + I(exper_c**2) + female + urban", data=d).fit()
pd.DataFrame({"raw experience": vif_table(m).values,
              "centred experience": vif_table(m_c).values},
             index=["educ", "exper", "exper²", "female", "urban"])

## 19.5 Omitted variable bias

In [ ]:
controls = "educ + exper + I(exper**2) + female"
long_ = smf.ols(f"lwage ~ {controls} + urban", data=d).fit()
short = smf.ols(f"lwage ~ {controls}", data=d).fit()
delta = smf.ols(f"urban ~ {controls}", data=d).fit()

print(f"long  regression, educ: {long_.params['educ']:.5f}")
print(f"short regression, educ: {short.params['educ']:.5f}")
print(f"long + beta_urban × delta_educ = {long_.params['educ']:.5f} + "
      f"{long_.params['urban']:.4f} × {delta.params['educ']:.4f} = "
      f"{long_.params['educ'] + long_.params['urban'] * delta.params['educ']:.5f}")

In [ ]:
rng = np.random.default_rng(seed=19)


def simulate_ovb(n=5_000, reps=500, true_return=0.08, ability_wage=0.15,
                 ability_school=2.0):
    """Short-regression slope of log wage on schooling when ability is omitted."""
    slopes = np.empty(reps)
    for r in range(reps):
        ability = rng.normal(0, 1, n)
        school = 11 + ability_school * ability + rng.normal(0, 4, n)
        lw = 8 + true_return * school + ability_wage * ability + rng.normal(0, 0.6, n)
        slopes[r] = np.cov(school, lw)[0, 1] / school.var(ddof=1)
    theory = true_return + ability_wage * ability_school / (ability_school ** 2 + 16)
    return slopes.mean(), theory


for a_w in [0.0, 0.10, 0.15, 0.25]:
    est, theory = simulate_ovb(ability_wage=a_w)
    print(f"ability effect on wage {a_w:.2f}: average estimate {est:.4f}, "
          f"formula {theory:.4f} (true return 0.08)")

## 19.6 Functional form: the RESET test

In [ ]:
from statsmodels.stats.diagnostic import linear_reset

r_log = linear_reset(m, power=3, use_f=True)
m_lvl = smf.ols(MINCER.replace("lwage", "earn"), data=d).fit()
r_lvl = linear_reset(m_lvl, power=3, use_f=True)
m_sq = smf.ols(MINCER + " + I(educ**2)", data=d).fit()
r_sq = linear_reset(m_sq, power=3, use_f=True)

pd.DataFrame({
    "RESET F": [r_log.fvalue, r_lvl.fvalue, r_sq.fvalue],
    "p-value": [r_log.pvalue, r_lvl.pvalue, r_sq.pvalue],
    "R²": [m.rsquared, m_lvl.rsquared, m_sq.rsquared],
}, index=["log earnings (Mincer)", "earnings in ₹ (levels)",
          "log earnings + schooling²"]).round(3)

In [ ]:
print(f"Schooling²: coefficient {m_sq.params['I(educ ** 2)']:.4f} "
      f"(t = {m_sq.tvalues['I(educ ** 2)']:.1f}); marginal return at 5 years "
      f"{m_sq.params['educ'] + 2 * 5 * m_sq.params['I(educ ** 2)']:.3f}, "
      f"at 15 years {m_sq.params['educ'] + 2 * 15 * m_sq.params['I(educ ** 2)']:.3f}")

## 19.7 Outliers and influential observations

In [ ]:
infl = m.get_influence()
cooks = infl.cooks_distance[0]
lev = infl.hat_matrix_diag
stud = infl.resid_studentized_internal
flag = cooks > 4 / len(d)
print(f"Leverage: mean {lev.mean():.5f} (= k/n), max {lev.max():.5f}")
print(f"Observations with Cook's D > 4/n: {flag.sum():,} ({flag.mean():.1%})")
print(f"Lowest reported salaries: {np.sort(d['earn'].unique())[:5]}")
print(f"Highest: {np.sort(d['earn'].unique())[-5:]}")
neg = m.resid[flag] < 0
print(f"Flagged with negative residuals: {neg.sum():,}; positive: {(~neg).sum():,}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.scatter(lev[~flag], stud[~flag], s=4, color="#9db7d5", alpha=0.5,
           label="Cook's D ≤ 4/n")
ax.scatter(lev[flag], stud[flag], s=6, color="#c55a11", alpha=0.7,
           label="Cook's D > 4/n")
ax.axhline(0, color="black", lw=0.6)
ax.set_xlabel("Leverage (diagonal of the hat matrix)")
ax.set_ylabel("Studentised residual")
ax.set_title("Influence: large residuals matter more at high leverage",
             loc="left", fontsize=13)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, markerscale=3, loc="lower right")
fig.text(0.01, -0.05, "Mincer regression of log monthly earnings. Source: "
         "author's calculation from PLFS 2023–24 unit-level data (NSO, MoSPI).",
         fontsize=8, color="grey")
plt.show()

In [ ]:
robust_checks = pd.DataFrame({
    "all workers": m.params,
    "drop Cook's D > 4/n": smf.ols(MINCER, data=d[~flag]).fit().params,
    "drop earnings < ₹2,000": smf.ols(MINCER, data=d[d["earn"] >= 2000]).fit().params,
    "median regression": smf.quantreg(MINCER, data=d).fit(q=0.5).params,
}).loc[["educ", "exper", "female", "urban"]].round(4)
robust_checks

## 19.8 Measurement error and attenuation

In [ ]:
x0, y0 = d["educ"].to_numpy(), d["lwage"].to_numpy()
var_x = x0.var()
b_true = np.cov(x0, y0)[0, 1] / var_x          # simple-regression slope, no noise

rows = []
for sd_e in [0, 1, 2, 3, 4]:
    xs = x0 + rng.normal(0, sd_e, len(x0))
    b = np.cov(xs, y0)[0, 1] / xs.var(ddof=1)
    rows.append({"noise SD (years)": sd_e, "estimated slope": b,
                 "theory β·λ": b_true * var_x / (var_x + sd_e ** 2),
                 "λ": var_x / (var_x + sd_e ** 2)})
atten = pd.DataFrame(rows).set_index("noise SD (years)")
atten.round(4)

## 19.9 Sample selection: whose wages do we see?

In [ ]:
w = p[p["sex"].isin([2]) & p["age"].between(15, 59).fillna(False)]
w = pd.DataFrame({
    "salaried": (w["acws"].isin([31]) & w["ern_reg"].gt(0).fillna(False)).astype(float),
    "lwage": np.log(w["ern_reg"].astype(float).where(w["ern_reg"] > 0)),
    "educ": w["form_edu"].astype(float),
    "age": w["age"].astype(float),
    "urban": (w["sec"] == 2).astype(float),
    "married": (w["marst"] == 2).astype(float),
    "kids05": w["kids05"].astype(float),
})
w["exper"] = (w["age"] - w["educ"] - 6).clip(lower=0)
del p                                                       # free memory
print(f"Women aged 15–59 in the sample: {len(w):,}; "
      f"with a salaried job and earnings: {int(w['salaried'].sum()):,} "
      f"({w['salaried'].mean():.1%}, unweighted)")

In [ ]:
probit = smf.probit("salaried ~ educ + exper + I(exper**2) + urban + married"
                    " + kids05", data=w).fit(disp=0)
print(probit.summary().tables[1])

In [ ]:
zg = probit.fittedvalues                         # the linear index z'γ
w["mills"] = stats.norm.pdf(zg) / stats.norm.cdf(zg)
ww = w[w["salaried"] == 1]

ols_w = smf.ols("lwage ~ educ + exper + I(exper**2) + urban", data=ww).fit()
heck = smf.ols("lwage ~ educ + exper + I(exper**2) + urban + mills", data=ww).fit()
pd.DataFrame({"OLS, salaried women": ols_w.params,
              "Heckman two-step": heck.params}).reindex(heck.params.index).round(4)

In [ ]:
print(f"Coefficient on the inverse Mills ratio: {heck.params['mills']:.4f} "
      f"(t = {heck.tvalues['mills']:.2f}, naive SE)")

## 19.10 A checklist

## Key ideas

## Exercises

## Data citation and terms of use

## Further reading